In [11]:
import polars as pl

# recode flagged data with probability of value (0: invalid, 0.5: uncertain, 1: valid)
#  reformat as dataframe with 'zzzztttt', 'installation', 'parameter', 'probability'
import polars as pl
year = 2022
df = pl.read_parquet(f"data/level2/{year}/vrxa00.parquet")

# select only data where at least one parameter was flagged
df = df.select(pl.col('zzzztttt'), 
          pl.col('^f_.*$'),).filter(pl.any_horizontal(pl.col('^f_.*$')>0))
print(df.describe())

# recode flags
mapper = {
    0: 1,
    1: 0,
    2: 0.5,
}
df = df.select(
    pl.all().replace(mapper)
)

# 
installations = {
    'tre200s0':'24036',
    'rre150z0':'24036',
    'ure200s0':'24036',
    'gor000z0':'24036',
    'uor200s0':'24036',
    'ua2200s0':'24037',
    'ta2200s0':'24037',
    'ta1200s0':'22718',
    'ra1150z0':'22718',
    'ua1200s0':'22718',
    'fkl010z0':'22718',
    'dkl010z0':'22718',
    'pa1stas0':'22718',
}

# reshape dataframe, filter data
value_vars = [name for name in df.columns if name.startswith('f_')]
df = df.melt(id_vars=['zzzztttt'], value_name='probability', value_vars=value_vars)
df = df.filter(pl.col('probability')<1)

# rename columns, save dataframe
df = df.with_columns(pl.col('variable').str.replace_all('f_', ''))
df = df.with_columns(pl.col('variable').replace(installations).alias('installation'))
df = df.select(pl.col('zzzztttt'), pl.col('installation'), pl.col('variable'), pl.col('probability'))
df.write_csv(f"data/level2/{year}/vrxa00_probabilities.csv")

shape: (9, 19)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ statistic ┆ zzzztttt  ┆ f_tre200s ┆ f_uor200s ┆ … ┆ f_ua2200s ┆ f_dkl010z ┆ f_ra1150z ┆ f_fkl010 │
│ ---       ┆ ---       ┆ 0         ┆ 0         ┆   ┆ 0         ┆ 0         ┆ 0         ┆ z1       │
│ str       ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ count     ┆ 1590      ┆ 1128.0    ┆ 1128.0    ┆ … ┆ 1590.0    ┆ 1582.0    ┆ 1582.0    ┆ 1588.0   │
│ null_coun ┆ 0         ┆ 462.0     ┆ 462.0     ┆ … ┆ 0.0       ┆ 8.0       ┆ 8.0       ┆ 2.0      │
│ t         ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ mean      ┆ null      ┆ 0.093972  ┆ 0.199468  ┆ … ┆ 0.0       ┆ 0.927307  